<cell_type>markdown</cell_type># Dask Out-of-Core S3 Validation (30 GB)

This notebook validates Dask out-of-core processing with a **30 GB dataset** stored in S3.

**Key demonstration**: Processing 60 million spans (30 GB expanded) with a cluster that has
only 2-8 GB of memory, proving true out-of-core streaming capabilities.

**Note**: This notebook is read-only from ConfigMap. To edit:
```python
import shutil
shutil.copy('/home/jovyan/sample-notebooks/Dask_S3_Validation.ipynb', '/home/jovyan/')
```

In [ ]:
# Imports
import os
import uuid
from datetime import datetime, timezone

# Air-gap: embed Bokeh JS (no cdn.bokeh.org). Keep kernel comms for zoom callbacks.
os.environ.setdefault("BOKEH_RESOURCES", "inline")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
import dask.dataframe as dd
from dask.distributed import Client

import holoviews as hv

# Holoviews 1.23 Image validation is picky when datashader supplies both
# coordinates *and* bounds (float edge mismatch → ValueError on DynamicMap).
hv.config.image_rtol = 1.0  # tolerate edge float noise; display extents still from data

hv.extension("bokeh", inline=True)

try:
    import panel as pn
    pn.extension()
    _HAS_PANEL = True
except Exception as _e:
    _HAS_PANEL = False
    print("panel unavailable:", _e, "— interactive zoom re-agg may be limited")


def show(obj, **kwargs):
    """Prefer Panel so DynamicMap zoom callbacks reach the kernel (no jupyter_bokeh)."""
    if _HAS_PANEL:
        return pn.panel(obj, **kwargs)
    return obj


def image_from_agg(agg, kdims, vdim="count"):
    """Build hv.Image from a datashader/xarray aggregate using *coordinates only*.

    Never pass separate ``bounds=`` — that is what triggers:
    ``Supplied Image bounds do not match the coordinates defined in the data``.
    """
    import xarray as xr

    if hasattr(agg, "compute"):
        agg = agg.compute()
    if isinstance(agg, xr.Dataset):
        # pick first data var
        name = list(agg.data_vars)[0]
        da = agg[name]
    else:
        da = agg
        name = getattr(da, "name", None) or vdim
    # datashader: dims are typically (y_dim, x_dim)
    ydim, xdim = da.dims[0], da.dims[1]
    ys = np.asarray(da.coords[ydim].values, dtype=np.float64)
    xs = np.asarray(da.coords[xdim].values, dtype=np.float64)
    arr = np.ascontiguousarray(np.asarray(da.values, dtype=np.float64))
    # Replace non-finite (empty bins) so Image ctor is happy
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.shape != (ys.size, xs.size):
        # transpose if needed
        if arr.shape == (xs.size, ys.size):
            arr = arr.T
        else:
            raise ValueError(f"agg shape {arr.shape} vs coords ys={ys.size} xs={xs.size}")
    return hv.Image(
        (xs, ys, arr),
        kdims=list(kdims),
        vdims=[vdim],
        rtol=1.0,
    )


print("Using AWS credential chain (IAM role / env vars / credentials file)")
print(f"hv inline=True  panel={_HAS_PANEL}  image_rtol={hv.config.image_rtol}")

# --- Cluster config (JupyterHub extraEnv from zarf/converge — no secret hard-codes) ---
import os, sys
for _p in ("/root/sample-notebooks", "/app", "/root"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from cluster_env import load_cluster_config, load_active_spans_ddf
    CFG = load_cluster_config()
    print(CFG.summary())
except ImportError as _e:
    print("cluster_env not mounted — using os.environ fallback:", _e)
    from types import SimpleNamespace
    _bucket = os.getenv("S3_BUCKET", "cyberphy")
    _ep = os.getenv("S3_ENDPOINT", "")
    _reg = os.getenv("AWS_REGION", os.getenv("S3_REGION", "us-east-1"))
    _path = os.getenv("OTEL_DATA_PATH", f"s3://{_bucket}/")
    _pref = os.getenv("OTEL_PREFIX", "")
    if not _pref and _path.startswith("s3://"):
        _rest = _path[5:].strip("/")
        if "/" in _rest:
            _pref = _rest.split("/", 1)[1]
    def _storage_options():
        o = {"anon": False, "key": os.getenv("AWS_ACCESS_KEY_ID") or None,
             "secret": os.getenv("AWS_SECRET_ACCESS_KEY") or None,
             "token": os.getenv("AWS_SESSION_TOKEN") or None}
        if _ep:
            o["client_kwargs"] = {"endpoint_url": _ep, "region_name": _reg}
            o["config_kwargs"] = {"s3": {"addressing_style": "path"}, "signature_version": "s3v4"}
        return o
    def _connect_dask():
        from dask.distributed import Client
        return Client(os.getenv("DASK_SCHEDULER_ADDRESS", "tcp://cybersec-dask-scheduler.dask.svc.cluster.local:8786"))
    CFG = SimpleNamespace(
        s3_bucket=_bucket, s3_endpoint=_ep, s3_region=_reg, otel_data_path=_path,
        otel_prefix=_pref,
        dask_scheduler=os.getenv("DASK_SCHEDULER_ADDRESS", "tcp://cybersec-dask-scheduler.dask.svc.cluster.local:8786"),
        use_dask=True, hdf5_profile=os.getenv("HDF5_PROFILE", "lab"),
        storage_options=_storage_options, connect_dask=_connect_dask,
        dataset_root_key=f"{_bucket}/{_pref}".strip("/"),
        spans_prefix_s3=f"s3://{_bucket}/{_pref}/spans/".replace("//spans", "/spans") if _pref else f"s3://{_bucket}/spans/",
        summary=lambda: f"fallback CFG bucket={_bucket} endpoint={_ep or 'AWS'} prefix={_pref}",
    )
    def load_active_spans_ddf(cfg=None, columns=None, require_dask=True):
        import dask.dataframe as dd, s3fs
        cfg = cfg or CFG
        fs = s3fs.S3FileSystem(**cfg.storage_options())
        spans = f"{cfg.dataset_root_key}/spans"
        files = []
        for pat in (f"{spans}/date=*/hour=*/*.parquet", f"{spans}/date=*/*.parquet", f"{spans}/shard=*/date=*/*.parquet"):
            hits = [h for h in (fs.glob(pat) or []) if str(h).endswith(".parquet")]
            if hits:
                files = hits; break
        if not files and fs.exists(spans):
            files = [e for e in fs.find(spans) if str(e).endswith(".parquet")]
        if not files:
            raise FileNotFoundError(f"no parquet under s3://{spans}/")
        uris = [f"s3://{f}" if not str(f).startswith("s3://") else f for f in files]
        return dd.read_parquet(uris, storage_options=cfg.storage_options(), columns=columns, engine="pyarrow")
    print(CFG.summary() if callable(CFG.summary) else CFG.summary)


## 1. Connect to Dask Cluster

In [ ]:
# Connect to in-cluster Dask (scheduler address from JupyterHub env / converge)
client = CFG.connect_dask() if hasattr(CFG, "connect_dask") and callable(CFG.connect_dask) else __import__("dask.distributed", fromlist=["Client"]).Client(CFG.dask_scheduler)
print(f"Dashboard: {client.dashboard_link}")
print(f"Workers: {len(client.scheduler_info().get('workers', {}))}")
client


## 2. Configure S3 Access

In [ ]:
# S3 via cluster_env / JupyterHub env (converge --creds-file → zarf → singleuser)
import s3fs
s3_opts = CFG.storage_options() if callable(getattr(CFG, "storage_options", None)) else CFG.storage_options
s3 = s3fs.S3FileSystem(**s3_opts)
bucket_name = CFG.s3_bucket
# Default prefix matches field panel OTEL_DATA_PATH (…/otel-notebook/), NOT validation-30gb
data_prefix = (getattr(CFG, "otel_prefix", None) or os.getenv("OTEL_PREFIX") or "otel-notebook").strip("/")
spans_uri = getattr(CFG, "spans_prefix_s3", None) or f"s3://{bucket_name}/{data_prefix}/spans/"
print(f"Endpoint: {CFG.s3_endpoint or 'AWS default'}")
print(f"Bucket:   {bucket_name}")
print(f"Prefix:   {data_prefix}")
print(f"Spans:    {spans_uri}")
print("  ⚠ Analytics default to this spans/ tree (partitioned).")
print("  ⚠ validation-30gb / validation-dask is OPTIONAL synthetic stress only.")
try:
    contents = s3.ls(bucket_name)
    print(f"Bucket listing (top): {contents[:12]}{'…' if len(contents) > 12 else ''}")
except Exception as e:
    print(f"Bucket list warning: {e}")


<cell_type>markdown</cell_type>## 3. Generate Large Synthetic Dataset (Idempotent)

We generate a **30 GB dataset** (expanded memory) to validate true out-of-core processing.
Each span includes additional attributes (~500 bytes/row in memory).

**Dataset Configuration:**
- Target: 30 GB expanded memory footprint
- Spans: ~60 million
- Partitions: 600 (100K spans each)
- Disk: ~6-8 GB (Parquet compressed with Snappy)

This dataset is typically **10-30× larger than cluster memory**, guaranteeing
that Dask must use streaming/out-of-core processing.

In [ ]:
# Data source selection — PATH config, not schema
# Field failure mode: loading s3://…/validation-30gb/*.parquet (empty) → columns=[]
# looks like "schema drift" but is bucket/path drift. Real data:
#   s3://$S3_BUCKET/$OTEL_PREFIX/spans/date=*/hour=*/*.parquet
import os

BYTES_PER_SPAN = 500
TOTAL_SPANS = int(os.getenv("VALIDATION_SPANS", "5000000"))
SPANS_PER_PARTITION = int(os.getenv("VALIDATION_SPANS_PER_PART", "100000"))
NUM_PARTITIONS = max(1, (TOTAL_SPANS + SPANS_PER_PARTITION - 1) // SPANS_PER_PARTITION)
NUM_SERVICES = 20
TARGET_MEMORY_BYTES = TOTAL_SPANS * BYTES_PER_SPAN
TARGET_MEMORY_GB = TARGET_MEMORY_BYTES / 1e9

scheduler_info = client.scheduler_info()
total_worker_memory = sum(w["memory_limit"] for w in scheduler_info["workers"].values()) or 1
num_workers = len(scheduler_info["workers"])
memory_ratio = TARGET_MEMORY_BYTES / total_worker_memory

# Default: USE active dataset under otel-notebook/spans (or OTEL_DATA_PATH).
# Opt-in synthetic path only: GENERATE_SYNTHETIC=1 (writes validation-dask/).
GENERATE_SYNTHETIC = os.getenv("GENERATE_SYNTHETIC", "0").lower() in ("1", "true", "yes")
USE_ACTIVE_SPANS = not GENERATE_SYNTHETIC  # inverted default vs old USE_ACTIVE_SPANS=0
# Allow explicit override
if os.getenv("USE_ACTIVE_SPANS", "").lower() in ("0", "false", "no"):
    USE_ACTIVE_SPANS = False
if os.getenv("USE_ACTIVE_SPANS", "").lower() in ("1", "true", "yes"):
    USE_ACTIVE_SPANS = True

validation_path = f"{bucket_name}/{data_prefix}/validation-dask"  # optional synthetic only
active_spans_key = f"{bucket_name}/{data_prefix}/spans"

print("=" * 60)
print("DATASET PATH (config — not schema)")
print("=" * 60)
print(f"  Active spans key:  s3://{active_spans_key}/")
print(f"  Synthetic path:    s3://{validation_path}/  (only if GENERATE_SYNTHETIC=1)")
print(f"  USE_ACTIVE_SPANS:  {USE_ACTIVE_SPANS}")
print(f"  GENERATE_SYNTHETIC:{GENERATE_SYNTHETIC}")
print(f"  Workers / mem:     {num_workers} / {total_worker_memory/1e9:.2f} GB")
if USE_ACTIVE_SPANS:
    print(f"\n✓ Will load ACTIVE spans under s3://{active_spans_key}/ (partitioned)")
    GENERATE_DATA = False
else:
    print(f"\n⚠ Synthetic mode — will use s3://{validation_path}/")
    try:
        existing_files = s3.glob(f"{validation_path}/**/*.parquet") or s3.glob(f"{validation_path}/*.parquet")
        if existing_files and len(existing_files) >= min(NUM_PARTITIONS, 10):
            print(f"  Existing synthetic files: {len(existing_files)}")
            GENERATE_DATA = False
        else:
            GENERATE_DATA = True
    except Exception as e:
        print(f"  No synthetic data yet: {e}")
        GENERATE_DATA = True


In [ ]:
def generate_spans(n_spans=100000, n_services=20):
    """Generate synthetic span data with rich attributes."""
    services = [f"service-{i:02d}" for i in range(n_services)]
    operations = [
        'GET /api/v1/users', 'POST /api/v1/orders', 'GET /health',
        'PUT /api/v1/users/{id}', 'DELETE /api/v1/orders/{id}',
        'db.query.select', 'db.query.insert', 'db.query.update',
        'cache.get', 'cache.set', 'cache.delete',
        'queue.publish', 'queue.consume',
        'grpc.client.call', 'grpc.server.handle',
    ]
    http_methods = ['GET', 'POST', 'PUT', 'DELETE', 'PATCH']
    status_codes_http = [200, 201, 204, 400, 401, 403, 404, 500, 502, 503]

    now = datetime.now(timezone.utc)
    base_ts = int(now.timestamp() * 1e9)

    # Generate random data
    n = n_spans
    service_idx = np.random.randint(0, n_services, n)

    return pd.DataFrame({
        # Core span fields
        'trace_id': [uuid.uuid4().hex for _ in range(n)],
        'span_id': [uuid.uuid4().hex[:16] for _ in range(n)],
        'parent_span_id': np.where(
            np.random.random(n) > 0.3,
            [uuid.uuid4().hex[:16] for _ in range(n)],
            ''
        ),
        'service_name': np.array(services)[service_idx],
        'operation_name': np.random.choice(operations, n),

        # Timing
        'start_time_unix_nano': base_ts - np.random.randint(0, 86400 * 1e9, n).astype(np.int64),
        'duration_ns': np.random.exponential(scale=50_000_000, size=n).astype(np.int64),

        # Status
        'status_code': np.random.choice(['OK', 'ERROR', 'UNSET'], n, p=[0.92, 0.05, 0.03]),
        'http_status_code': np.random.choice(status_codes_http, n, p=[0.5, 0.1, 0.05, 0.1, 0.05, 0.05, 0.05, 0.05, 0.025, 0.025]),
        'http_method': np.random.choice(http_methods, n, p=[0.5, 0.25, 0.1, 0.1, 0.05]),

        # Additional attributes for richer data
        'user_id': [f"user-{np.random.randint(1, 10000):05d}" for _ in range(n)],
        'request_id': [uuid.uuid4().hex for _ in range(n)],
        'host': np.array([f"{services[i]}-{np.random.randint(1,10)}.cluster.local" for i in service_idx]),
        'db_statement': np.where(
            np.random.random(n) > 0.7,
            [f"SELECT * FROM table_{np.random.randint(1,100)} WHERE id = {np.random.randint(1,1000000)}" for _ in range(n)],
            ''
        ),
        'error_message': np.where(
            np.random.random(n) > 0.95,
            np.random.choice(['Connection timeout', 'Rate limit exceeded', 'Internal error', 'Not found', 'Permission denied'], n),
            ''
        ),
    })

if GENERATE_DATA:
    import time as time_module
    
    print(f"Generating {TOTAL_SPANS:,} spans across {NUM_PARTITIONS} partitions...")
    print(f"  {SPANS_PER_PARTITION:,} spans per partition")
    print(f"  {NUM_SERVICES} services")
    print(f"  Estimated time: {NUM_PARTITIONS * 2 / 60:.0f}-{NUM_PARTITIONS * 4 / 60:.0f} minutes")
    print()
    
    start_time = time_module.time()
    bytes_written = 0

    for partition in range(NUM_PARTITIONS):
        df = generate_spans(n_spans=SPANS_PER_PARTITION, n_services=NUM_SERVICES)

        # Convert to PyArrow table
        table = pa.Table.from_pandas(df, preserve_index=False)

        # Write to S3
        output_path = f"s3://{validation_path}/partition_{partition:04d}.parquet"
        pq.write_table(table, output_path, filesystem=s3, compression='snappy')
        
        # Track progress
        file_size = s3.info(f"{validation_path}/partition_{partition:04d}.parquet")['size']
        bytes_written += file_size
        
        # Progress update every 50 partitions
        if (partition + 1) % 50 == 0:
            elapsed = time_module.time() - start_time
            rate = (partition + 1) / elapsed
            eta = (NUM_PARTITIONS - partition - 1) / rate
            print(f"  [{partition + 1:4d}/{NUM_PARTITIONS}] "
                  f"{bytes_written/1e9:.2f} GB written, "
                  f"{rate:.1f} partitions/sec, "
                  f"ETA: {eta/60:.1f} min")

    elapsed = time_module.time() - start_time
    
    # Print final stats
    files = s3.glob(f"{validation_path}/*.parquet")
    total_size = sum(s3.info(f)['size'] for f in files)
    
    print(f"\n{'='*60}")
    print("DATA GENERATION COMPLETE")
    print(f"{'='*60}")
    print(f"  Files:       {len(files)}")
    print(f"  Total size:  {total_size/1024/1024/1024:.2f} GB (compressed)")
    print(f"  Spans:       {TOTAL_SPANS:,}")
    print(f"  Time:        {elapsed/60:.1f} minutes")
    print(f"  Throughput:  {TOTAL_SPANS/elapsed:,.0f} spans/sec")
else:
    print("Using existing data")

## 4. Load Data with Dask (Lazy)

In [ ]:
# Load spans from OTEL_Data_Generator layout (pyarrow notebook):
#   s3://$BUCKET/$PREFIX/spans/date=YYYY-MM-DD/*.parquet
# (date= only — NO hour=). Not validation-30gb; not date=*/**/*.

from cluster_env import load_active_spans_ddf, list_span_parquet_keys

spans_key = f"{bucket_name}/{data_prefix}/spans"
print(f"Listing parquet under s3://{spans_key}/ …")
print("  expected layout: date=YYYY-MM-DD/batch_*.parquet  (OTEL generator)")
keys = list_span_parquet_keys(CFG)
print(f"  found {len(keys)} files")
if keys:
    print(f"  sample: {keys[:5]}")
if not keys:
    raise FileNotFoundError(
        f"No parquet under s3://{spans_key}/.\n"
        f"Generator writes: s3://{spans_key}/date=YYYY-MM-DD/*.parquet"
    )

ddf = load_active_spans_ddf(CFG)
print(f"Partitions: {ddf.npartitions}")
print(f"Columns ({len(ddf.columns)}): {list(ddf.columns)}")
assert list(ddf.columns), "columns=[] — path/credentials problem, not schema drift"
print("⚡ LAZY — compute only aggregates on the cluster")
ddf


## 4. Load data with Dask (lazy)

**OTEL_Data_Generator layout** (this is what the arrow notebook writes):

```text
s3://$S3_BUCKET/$PREFIX/spans/date=YYYY-MM-DD/batch_*.parquet
```

`date=` only — **no `hour=`**. Production 1TB trees may add `hour=`; discovery handles both.

Do **not** use `validation-30gb` or globs with `**`.


In [ ]:
# Re-load (safe re-run) — same rules as above
from cluster_env import load_active_spans_ddf

if USE_ACTIVE_SPANS:
    data_uri = f"s3://{bucket_name}/{data_prefix}/spans/"
    ddf = load_active_spans_ddf(CFG)
else:
    data_uri = f"s3://{validation_path}/"
    print(f"SYNTHETIC mode: {data_uri}")
    ddf = dd.read_parquet(
        f"{data_uri}*.parquet",
        storage_options=s3_opts,
        engine="pyarrow",
    )

print(f"URI:        {data_uri if USE_ACTIVE_SPANS else data_uri}")
print(f"Partitions: {ddf.npartitions}")
print(f"Columns:    {list(ddf.columns)}")
if not list(ddf.columns):
    raise RuntimeError(
        "columns=[] — wrong path or bad credentials. "
        f"Expected s3://{bucket_name}/{data_prefix}/spans/date=*/hour=*/*.parquet. "
        "Do not use date=*/**/*.parquet globs."
    )
ddf


In [ ]:
%%time
# Predicate + projection via Dask/Arrow engine on workers (not kernel to_table)
print("=" * 60)
print("DASK FILTER + AGG (distributed)")
print("=" * 60)

cols = [c for c in (
    "trace_id", "service_name", "operation_name", "duration_ns",
    "error_message", "status_code", "span_id",
) if c in ddf.columns]
ddf_err = ddf[cols] if cols else ddf

if "status_code" in ddf_err.columns:
    # status may be string OK/ERROR or int
    sc = ddf_err["status_code"]
    try:
        mask = (sc == "ERROR") | (sc == 2) | (sc.astype(str) == "ERROR")
    except Exception:
        mask = sc == "ERROR"
    ddf_err = ddf_err[mask]

if "service_name" in ddf_err.columns:
    agg_map = {}
    if "trace_id" in ddf_err.columns:
        agg_map["trace_id"] = "count"
    elif "span_id" in ddf_err.columns:
        agg_map["span_id"] = "count"
    if "duration_ns" in ddf_err.columns:
        agg_map["duration_ns"] = ["mean", "max"]
    if agg_map:
        error_by_service = ddf_err.groupby("service_name").agg(agg_map).compute()
        print(error_by_service.head(20))
    else:
        print(ddf_err.groupby("service_name").size().compute().head(20))
else:
    print("No service_name column — partition count only:", ddf.npartitions)
    print("Row estimate (expensive on huge data — optional): skip")


In [ ]:
%%time
# Dask filters on the ALREADY-BOUND ddf (same path as load cell)
print("=" * 60)
print("DASK FILTER + AGG on loaded path")
print("=" * 60)
if not list(ddf.columns):
    raise RuntimeError("ddf has no columns — fix path load first (not schema)")

want = ["trace_id", "service_name", "operation_name", "duration_ns", "status_code", "span_id"]
cols = [c for c in want if c in ddf.columns]
ddf_f = ddf[cols] if cols else ddf

if "status_code" in ddf_f.columns:
    sc = ddf_f["status_code"]
    try:
        ddf_f = ddf_f[(sc == "ERROR") | (sc.astype(str) == "ERROR") | (sc == 2)]
    except Exception:
        pass

if "service_name" in ddf_f.columns:
    if "duration_ns" in ddf_f.columns:
        out = ddf_f.groupby("service_name").agg({"duration_ns": ["count", "mean", "max"]}).compute()
    else:
        out = ddf_f.groupby("service_name").size().compute()
    print(out.head(15) if hasattr(out, "head") else out)
else:
    print("Available columns:", list(ddf.columns))
    print("(service_name missing — check you loaded span parquet, not an empty prefix)")


In [ ]:
# SCANNER API: Fine-grained batch iteration for streaming processing
# This is the most memory-efficient way to process large datasets

print("="*60)
print("ARROW SCANNER: Streaming Batch Processing")
print("="*60)

# Create scanner with batch size control
scanner = dataset.scanner(
    columns=['service_name', 'duration_ns', 'status_code'],
    filter=(ds.field('http_status_code') >= 500),  # Server errors only
    batch_size=100_000,  # 100K rows per batch
)

# Process in batches - never loads full dataset
batch_count = 0
total_rows = 0
duration_sum = 0

for batch in scanner.to_batches():
    batch_count += 1
    total_rows += len(batch)
    # Aggregate within batch (zero-copy access to Arrow arrays)
    duration_sum += batch.column('duration_ns').to_numpy().sum()

avg_duration_ms = (duration_sum / total_rows / 1e6) if total_rows > 0 else 0

print(f"\nProcessed {batch_count} batches ({total_rows:,} rows)")
print(f"Average latency for HTTP 5xx errors: {avg_duration_ms:.2f} ms")
print(f"\n✓ Memory-efficient: only one batch in memory at a time")
print(f"  Each batch: ~{100_000 * 50 / 1024 / 1024:.1f} MB")
print(f"  Full dataset: {TARGET_MEMORY_GB} GB")
print(f"  Memory savings: {TARGET_MEMORY_GB * 1000 / (100_000 * 50 / 1024 / 1024):.0f}×")

## 5. Compute Aggregations (Distributed)

In [ ]:
%%time
# This triggers distributed computation across all workers
# Watch the Dask dashboard to see tasks being distributed!
total_spans = len(ddf)
print(f"✓ Total spans: {total_spans:,}")
print(f"  Processing {total_spans:,} rows across {ddf.npartitions} partitions")

In [ ]:
%%time
# Multi-dimensional aggregation on the loaded ddf (correct path already bound)
# If this KeyErrors service_name with columns=[], the load path was wrong
# (e.g. validation-30gb empty) — not a schema change.
print("Columns:", list(ddf.columns))
if "service_name" not in ddf.columns:
    raise RuntimeError(
        f"service_name not in columns={list(ddf.columns)}. "
        f"Path config drift: re-run the load cell against "
        f"s3://{bucket_name}/{data_prefix}/spans/ (not validation-30gb)."
    )

# Prefer span_id (OTEL) or fall back
count_col = "span_id" if "span_id" in ddf.columns else (
    "trace_id" if "trace_id" in ddf.columns else None
)
agg = {}
if count_col:
    agg[count_col] = "count"
if "duration_ns" in ddf.columns:
    agg["duration_ns"] = ["mean", "max", "min", "std"]
if "http_status_code" in ddf.columns:
    agg["http_status_code"] = ["mean"]

service_stats = ddf.groupby("service_name").agg(agg).compute()
# Flatten multiindex columns if needed
if isinstance(service_stats.columns, pd.MultiIndex):
    service_stats.columns = [
        "_".join(str(x) for x in col if x != "").strip("_")
        for col in service_stats.columns.values
    ]
# Normalize names
rename = {}
for c in list(service_stats.columns):
    if c.endswith("_count") or c == count_col:
        rename[c] = "count"
    if "mean" in c and "duration" in c:
        rename[c] = "mean_duration_ns"
    if "max" in c and "duration" in c:
        rename[c] = "max_duration_ns"
    if "min" in c and "duration" in c:
        rename[c] = "min_duration_ns"
    if "std" in c and "duration" in c:
        rename[c] = "std_duration_ns"
service_stats = service_stats.rename(columns=rename)
if "mean_duration_ns" in service_stats.columns:
    service_stats["mean_duration_ms"] = service_stats["mean_duration_ns"] / 1e6
if "max_duration_ns" in service_stats.columns:
    service_stats["max_duration_ms"] = service_stats["max_duration_ns"] / 1e6
if "count" in service_stats.columns:
    service_stats = service_stats.sort_values("count", ascending=False)
print(f"✓ Aggregated into {len(service_stats)} service groups")
cols_show = [c for c in ("count", "mean_duration_ms", "max_duration_ms", "std_duration_ns") if c in service_stats.columns]
service_stats[cols_show] if cols_show else service_stats


In [ ]:
%%time
# Error rate by service - OPTIMIZED (no apply/lambda)
# Count errors and total per service, then compute ratio
import dask

error_counts = ddf[ddf['status_code'] == 'ERROR'].groupby('service_name').size()
total_counts = ddf.groupby('service_name').size()

# Compute both in parallel
error_counts, total_counts = dask.compute(error_counts, total_counts)

# Calculate error rate
error_rate = (error_counts / total_counts * 100).fillna(0).sort_values(ascending=False)

print("Error rate by service (%)")
error_rate

## 6. Visualizations with HoloViews

In [ ]:
# Sample for visualization (compute a subset)
sample_df = ddf.sample(frac=0.1).compute()
print(f"Sample size: {len(sample_df):,} spans")

In [ ]:
# Duration distribution by service
sample_df['duration_ms'] = sample_df['duration_ns'] / 1e6

hv.BoxWhisker(sample_df, kdims='service_name', vdims='duration_ms').opts(
    width=800, height=400,
    title='Latency Distribution by Service',
    ylabel='Duration (ms)',
    box_fill_color=hv.dim('service_name').categorize(dict(zip(
        sample_df['service_name'].unique(),
        hv.Cycle('Category10').values
    )))
)

In [ ]:
# Span count by service (bar chart)
counts = sample_df.groupby('service_name').size().reset_index(name='count')

hv.Bars(counts, kdims='service_name', vdims='count').opts(
    width=600, height=400,
    title='Span Count by Service',
    color='service_name',
    cmap='Category10',
    xrotation=45,
)

In [ ]:
# Status distribution
status_counts = sample_df.groupby(['service_name', 'status_code']).size().reset_index(name='count')

hv.Bars(status_counts, kdims=['service_name', 'status_code'], vdims='count').opts(
    width=800, height=400,
    title='Status Code Distribution by Service',
    xrotation=45,
    color='status_code',
    cmap={'OK': 'green', 'ERROR': 'red', 'UNSET': 'gray'},
)

## 6b. Hierarchical Drill-Down + Dask Density

**Air-gap display:** `hv.extension('bokeh', inline=True)` embeds Bokeh JS (no CDN).
Interactive zoom callbacks use **Panel** (ipywidgets comms) because `jupyter_bokeh`
is not in the image.

| View | Data path | Zoom behavior |
|------|-----------|---------------|
| Service latency heatmap + hist | pandas `drill_df` sample | Client-side; hist filters on pan/zoom |
| Overview + detail | pandas `drill_df` | Detail re-aggregates hierarchy by range |
| **Dask density** | persisted Dask frame + `datashader.Canvas` | **Re-aggregates on workers** each zoom |

The density plot intentionally avoids `hv.operation.datashader.rasterize` DynamicMaps,
which raise `ValueError: Supplied Image bounds do not match the coordinates…` on
Holoviews 1.23. Images are rebuilt from coordinate arrays only.


In [ ]:
# Import drill-down visualization utilities
from holoviews.operation.datashader import rasterize, datashade
from holoviews.streams import RangeXY, RangeX

# Prepare data with proper datetime column
drill_df = sample_df.copy()
drill_df['timestamp'] = pd.to_datetime(drill_df['start_time_unix_nano'], unit='ns')
drill_df['duration_ms'] = drill_df['duration_ns'] / 1e6

print(f"Prepared {len(drill_df):,} spans for drill-down visualization")
print(f"Time range: {drill_df['timestamp'].min()} to {drill_df['timestamp'].max()}")

In [ ]:
# Service Latency Heatmap + linked histogram (client-side sample)
# Built as hv.Image from a dense pivot (coords only) to avoid HeatMap/Image
# bounds-validation errors on Holoviews 1.23 DynamicMap layouts.

from holoviews.streams import RangeXY

def create_latency_heatmap(df):
    df = df.copy()
    df["time_bucket"] = df["timestamp"].dt.floor("1min")
    pivot = df.pivot_table(
        index="service_name",
        columns="time_bucket",
        values="duration_ms",
        aggfunc="mean",
    ).sort_index()
    # Dense float grid; axes are integer indices (always regularly sampled)
    arr = np.nan_to_num(pivot.to_numpy(dtype=np.float64), nan=0.0)
    n_s, n_t = arr.shape
    if n_s == 0 or n_t == 0:
        return hv.Image((np.array([0.0]), np.array([0.0]), np.zeros((1, 1)))).opts(
            title="Service Latency Heatmap (empty)"
        )
    xs = np.arange(n_t, dtype=np.float64)          # time bucket index
    ys = np.arange(n_s, dtype=np.float64)          # service index
    # stash labels for hover/title
    create_latency_heatmap._tlabels = list(pivot.columns.astype(str))
    create_latency_heatmap._slabels = list(pivot.index.astype(str))
    create_latency_heatmap._tvals = list(pivot.columns)
    img = hv.Image(
        (xs, ys, arr),
        kdims=["time_idx", "service_idx"],
        vdims=["mean_latency"],
        rtol=1.0,
    )
    return img.opts(
        title="Service Latency Heatmap (zoom/pan → filters histogram)",
        colorbar=True,
        cmap="RdYlGn_r",
        default_tools=["pan", "wheel_zoom", "box_zoom", "reset", "hover"],
        active_tools=["wheel_zoom"],
        width=700,
        height=400,
        xlabel="time bucket index",
        ylabel="service index",
    )


heatmap = create_latency_heatmap(drill_df)
print(
    f"Heatmap grid services×buckets = "
    f"{len(getattr(create_latency_heatmap, '_slabels', []))}"
    f"×{len(getattr(create_latency_heatmap, '_tlabels', []))}"
)

range_stream = RangeXY(source=heatmap)


def update_histogram(x_range, y_range):
    """Filter drill_df by selected time-bucket indices on the Image x-axis."""
    filtered = drill_df
    tvals = getattr(create_latency_heatmap, "_tvals", None)
    if x_range and x_range[0] is not None and tvals:
        i0 = max(0, int(np.floor(x_range[0])))
        i1 = min(len(tvals) - 1, int(np.ceil(x_range[1])))
        if i1 < i0:
            i0, i1 = i1, i0
        t0, t1 = tvals[i0], tvals[i1]
        mask = (drill_df["timestamp"] >= t0) & (drill_df["timestamp"] <= t1 + pd.Timedelta("1min"))
        filtered = drill_df[mask]
    if filtered.empty:
        return hv.Histogram((np.array([0.0, 1.0]), np.array([0.0]))).opts(
            title="Latency Distribution (No data)", width=350, height=350
        )
    frequencies, edges = np.histogram(filtered["duration_ms"].to_numpy(), bins=50)
    return hv.Histogram((edges, frequencies)).opts(
        title=f"Latency Distribution (n={len(filtered):,})",
        xlabel="Latency (ms)",
        ylabel="Count",
        width=350,
        height=350,
        fill_color="#3182bd",
    )


histogram = hv.DynamicMap(update_histogram, streams=[range_stream])
linked = (heatmap + histogram).opts(shared_axes=False)
show(linked)


In [ ]:
# Overview + Detail View with Hierarchy-Aware Aggregation
# The detail view changes aggregation level based on zoom

def determine_agg_level(time_range_seconds):
    """Determine aggregation level from visible time range."""
    if time_range_seconds > 3600:    # > 1 hour
        return 'service_group', '10min'   # Group services, 10-min buckets
    elif time_range_seconds > 300:   # > 5 minutes
        return 'service', '1min'          # Individual services, 1-min buckets
    else:                            # < 5 minutes
        return 'host', '10s'              # Host level, 10-sec buckets

def create_overview(df):
    """Create overview area chart showing total request volume."""
    hourly = df.groupby(df['timestamp'].dt.floor('10min')).agg({
        'duration_ms': ['mean', 'count']
    }).reset_index()
    hourly.columns = ['time', 'mean_latency', 'count']
    
    area = hv.Area(hourly, kdims=['time'], vdims=['count'])
    return area.opts(
        height=120, width=900,
        alpha=0.7, color='#1f77b4',
        title='Overview: Request Volume (drag to zoom detail view)',
        tools=['xbox_select'],
    )

def create_detail(df, x_range):
    """Create detail view with hierarchy-aware aggregation."""
    if x_range is None or x_range[0] is None:
        # Default view
        time_range_sec = 3600
        filtered = df
    else:
        start, end = pd.Timestamp(x_range[0]), pd.Timestamp(x_range[1])
        time_range_sec = (end - start).total_seconds()
        filtered = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)]
    
    if filtered.empty:
        return hv.Scatter([]).opts(title='Detail (No data in range)')
    
    # Determine aggregation level
    group_col, time_bucket = determine_agg_level(time_range_sec)
    
    if group_col == 'service_group':
        # Group services into clusters (e.g., service-0X -> group-0)
        filtered = filtered.copy()
        filtered['_group'] = filtered['service_name'].str.extract(r'service-(\d)')[0].apply(
            lambda x: f'group-{x}' if pd.notna(x) else 'other'
        )
        group_by = '_group'
    elif group_col == 'host':
        group_by = 'host'
    else:
        group_by = 'service_name'
    
    # Aggregate
    filtered = filtered.copy()
    filtered['_bucket'] = filtered['timestamp'].dt.floor(time_bucket)
    
    agg = filtered.groupby([group_by, '_bucket']).agg({
        'duration_ms': 'mean',
    }).reset_index()
    agg.columns = ['group', 'time', 'latency']
    
    # Create scatter plot
    scatter = hv.Scatter(agg, kdims=['time'], vdims=['latency', 'group'])
    
    return scatter.opts(
        title=f'Detail: {group_col} level, {time_bucket} buckets (n={len(agg):,})',
        color='group',
        cmap='Category20',
        width=900, height=350,
        tools=['hover'],
        size=8,
        alpha=0.7,
    )

# Create the linked overview + detail
overview = create_overview(drill_df)
range_x = RangeX(source=overview)

detail = hv.DynamicMap(
    lambda x_range: create_detail(drill_df, x_range),
    streams=[range_x]
)

# Stack vertically
(overview + detail).cols(1).opts(shared_axes=False)

In [ ]:
# Dask-backed density heatmap: zoom/pan re-aggregates on the *cluster*
# ---------------------------------------------------------------------------
# Avoid holoviews.operation.datashader.rasterize DynamicMap → Image bounds bug
# (ValueError: Supplied Image bounds do not match the coordinates...).
# Instead: datashader.Canvas on a persisted Dask frame → hv.Image(coords only).

import datashader as ds
from holoviews.streams import RangeXY, PlotSize

DASK_VIZ_FRAC = float(os.getenv("DASK_VIZ_FRAC", "0.1"))

print("Building lazy Dask view columns (no full client .compute of all rows)…")
ddf_pts = ddf.assign(
    timestamp_numeric=ddf["start_time_unix_nano"] / 1e9,
    duration_ms=ddf["duration_ns"] / 1e6,
)[["timestamp_numeric", "duration_ms"]]
if 0 < DASK_VIZ_FRAC < 1.0:
    ddf_pts = ddf_pts.sample(frac=DASK_VIZ_FRAC, random_state=0)

print("Persisting partitions on Dask workers (watch dashboard)…")
ddf_pts = ddf_pts.persist()
try:
    from dask.distributed import wait as dask_wait
    dask_wait(ddf_pts)
except Exception:
    pass

# Global extents (two small scalar tasks)
_x0, _x1, _y0, _y1 = dd.compute(
    ddf_pts["timestamp_numeric"].min(),
    ddf_pts["timestamp_numeric"].max(),
    ddf_pts["duration_ms"].min(),
    ddf_pts["duration_ms"].max(),
)
# pad zero-width ranges
if not np.isfinite(_x0) or not np.isfinite(_x1) or _x1 <= _x0:
    _x0, _x1 = 0.0, 1.0
if not np.isfinite(_y0) or not np.isfinite(_y1) or _y1 <= _y0:
    _y0, _y1 = 0.0, 1.0
_dx = (_x1 - _x0) * 0.01 or 1.0
_dy = (_y1 - _y0) * 0.01 or 1.0
_x0, _x1 = float(_x0 - _dx), float(_x1 + _dx)
_y0, _y1 = float(_y0 - _dy), float(_y1 + _dy)

n_est = int(ddf_pts.shape[0].compute())
print(f"Dask points ready: ~{n_est} rows  partitions={ddf_pts.npartitions}")
print(f"extents x=[{_x0:.3g},{_x1:.3g}]  y=[{_y0:.3g},{_y1:.3g}]")


def dask_density_image(x_range, y_range, width=900, height=400, scale=1.0):
    """Canvas.points on Dask → hv.Image with coordinates only (no bounds=)."""
    w = max(int(width or 900), 2)
    h = max(int(height or 400), 2)
    xr = x_range if x_range and x_range[0] is not None else (_x0, _x1)
    yr = y_range if y_range and y_range[0] is not None else (_y0, _y1)
    # guard inverted / zero ranges
    x0, x1 = float(min(xr[0], xr[1])), float(max(xr[0], xr[1]))
    y0, y1 = float(min(yr[0], yr[1])), float(max(yr[0], yr[1]))
    if x1 <= x0:
        x1 = x0 + 1.0
    if y1 <= y0:
        y1 = y0 + 1.0

    cvs = ds.Canvas(plot_width=w, plot_height=h, x_range=(x0, x1), y_range=(y0, y1))
    # This is the Dask compute: workers aggregate only the visible bins
    agg = cvs.points(ddf_pts, "timestamp_numeric", "duration_ms", ds.count())
    img = image_from_agg(
        agg,
        kdims=["timestamp_numeric", "duration_ms"],
        vdim="count",
    )
    return img.opts(
        cmap="fire",
        colorbar=True,
        width=w,
        height=h,
        xlabel="Time (Unix seconds)",
        ylabel="Latency (ms)",
        title=f"Dask density (~{n_est} pts, frac={DASK_VIZ_FRAC}) — zoom re-aggs on workers",
        default_tools=["pan", "wheel_zoom", "box_zoom", "reset", "hover"],
        active_tools=["wheel_zoom"],
        framewise=True,   # allow axis ranges to follow stream
        # keep data ranges free so RangeXY can update
    )


# Streams: RangeXY + PlotSize. Seed ranges so first paint is valid.
_range = RangeXY(x_range=(_x0, _x1), y_range=(_y0, _y1))
_size = PlotSize(width=900, height=400)
density = hv.DynamicMap(dask_density_image, streams=[_range, _size])

# Attach RangeXY to the DynamicMap itself so wheel-zoom updates the stream
_range.source = density

print(
    "Zoom/pan and watch the Dask dashboard for new tasks. "
    "If the dashboard is idle, callbacks are not reaching the kernel."
)
show(density, width=920, height=440)


<cell_type>markdown</cell_type>## 7. Out-of-Core Stress Test (30 GB)

This section demonstrates that Dask processes data **much larger than worker memory** using streaming aggregation.
With a 30 GB dataset and typical 2-8 GB cluster memory, this is a 4-15× memory challenge.

Each worker processes partitions sequentially, computing partial results and releasing memory.

In [ ]:
import time

def show_worker_memory(label=""):
    """Display worker memory utilization."""
    info = client.scheduler_info()
    print(f"\n{'='*60}")
    print(f"Worker Memory Utilization {label}")
    print(f"{'='*60}")
    
    total_used = 0
    total_limit = 0
    
    for worker_id, worker in sorted(info['workers'].items()):
        mem_used = worker.get('metrics', {}).get('memory', 0)
        mem_limit = worker['memory_limit']
        pct = (mem_used / mem_limit * 100) if mem_limit else 0
        worker_short = worker_id.split('/')[-1][:25]
        bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
        print(f"  {worker_short:25} [{bar}] {mem_used/1e9:.2f}GB / {mem_limit/1e9:.2f}GB ({pct:5.1f}%)")
        total_used += mem_used
        total_limit += mem_limit
    
    print(f"\n  TOTAL: {total_used/1e9:.2f}GB / {total_limit/1e9:.2f}GB ({total_used/total_limit*100:.1f}%)")
    return total_used, total_limit

# Show baseline memory and dataset ratio
baseline_used, baseline_limit = show_worker_memory("(baseline)")

print(f"\n{'='*60}")
print("OUT-OF-CORE CHALLENGE: 30 GB DATASET")
print(f"{'='*60}")
print(f"\n  Dataset expanded size:   {TARGET_MEMORY_GB} GB")
print(f"  Total worker memory:     {baseline_limit/1e9:.2f} GB")
print(f"  Ratio (dataset/workers): {TARGET_MEMORY_BYTES/baseline_limit:.1f}×")
print(f"\n  ⚡ Dataset is {TARGET_MEMORY_BYTES/baseline_limit:.1f}× larger than cluster memory!")
print(f"     Dask MUST use out-of-core streaming to process this.")

In [ ]:
%%time
# Complex multi-key aggregation — only columns that exist on the loaded path
print(f"Columns available: {list(ddf.columns)}")
need = {"service_name"}
missing = need - set(ddf.columns)
if missing:
    raise RuntimeError(
        f"Missing {missing} — empty/wrong load path (columns={list(ddf.columns)}). "
        f"Use s3://…/{data_prefix}/spans/ not validation-30gb."
    )

work = ddf
if "http_status_code" in ddf.columns:
    work = ddf.assign(is_http_error=(ddf["http_status_code"] >= 400).astype(int))
    err_col = "is_http_error"
else:
    err_col = None

group_cols = [c for c in ("service_name", "operation_name", "http_method", "name") if c in work.columns]
if len(group_cols) < 1:
    group_cols = ["service_name"]

agg = {}
if "span_id" in work.columns:
    agg["span_id"] = "count"
elif "trace_id" in work.columns:
    agg["trace_id"] = "count"
if "duration_ns" in work.columns:
    agg["duration_ns"] = ["mean", "max", "std"]
if err_col:
    agg[err_col] = "sum"

print(f"groupby={group_cols} agg={list(agg)}")
complex_stats = work.groupby(group_cols).agg(agg).compute()
if isinstance(complex_stats.columns, pd.MultiIndex):
    complex_stats.columns = [
        "_".join(str(x) for x in col if x != "").strip("_")
        for col in complex_stats.columns.values
    ]
print(complex_stats.head(15))
complex_stats.head(15)


In [ ]:
# Check memory AFTER complex aggregation
# Key insight: memory should still be bounded even after processing 30 GB dataset
post_used, post_limit = show_worker_memory("(after complex aggregation)")

print("\n" + "="*60)
print("OUT-OF-CORE VERIFICATION: 30 GB DATASET")
print("="*60)

# Calculate memory headroom
max_worker_pct = max(
    w.get('metrics', {}).get('memory', 0) / w['memory_limit'] * 100
    for w in client.scheduler_info()['workers'].values()
)

print(f"""
Dataset size:          {TARGET_MEMORY_GB} GB ({TOTAL_SPANS:,} spans)
Cluster memory:        {post_limit/1e9:.1f} GB
Dataset/Memory ratio:  {TARGET_MEMORY_BYTES/post_limit:.1f}×
Peak worker memory:    {max_worker_pct:.1f}%
Memory stayed bounded: {'✓ YES' if max_worker_pct < 90 else '✗ NO'}

How out-of-core processing works:
  1. Dask reads partitions lazily from S3 (one at a time per task)
  2. Each partition ({SPANS_PER_PARTITION:,} spans, ~50 MB) is processed
  3. Partial aggregation results accumulated, raw data released
  4. Only final aggregated results stay in memory

Result: Processed {TARGET_MEMORY_GB} GB dataset using {post_limit/1e9:.1f} GB cluster!
        That's {TARGET_MEMORY_BYTES/post_limit:.0f}× more data than available RAM.
""")

## 8. Cleanup

In [ ]:
# Optionally delete test data
# Uncomment to clean up:
# s3.rm(validation_path, recursive=True)
# print(f"Deleted: {validation_path}")

In [ ]:
# Close client
# client.close()

<cell_type>markdown</cell_type>---

## Validation Summary

This notebook validated **Dask out-of-core processing** with a **30 GB dataset**:

| Metric | Value |
|--------|-------|
| Dataset Size | **30 GB** (expanded memory) |
| Total Spans | 60,000,000 |
| Partitions | 600 |
| Disk Size | ~6-8 GB (Parquet/Snappy) |
| Services | 20 |

### Key Validation: 30 GB Out-of-Core Processing

The 30 GB dataset is typically **10-30× larger** than cluster memory:
- With 2 workers × 2 GB = 4 GB cluster → **7.5× ratio**
- With 4 workers × 2 GB = 8 GB cluster → **3.75× ratio**
- With 2 workers × 1 GB = 2 GB cluster → **15× ratio**

This **guarantees** Dask must use streaming/out-of-core processing - the data
simply cannot fit in memory.

### Capabilities Demonstrated

1. **Dask Cluster Connection** - Connected to distributed scheduler with multiple workers
2. **S3 Access via IAM** - Read/write to S3 using AWS IAM role credentials
3. **30 GB Dataset Generation** - 60M spans across 600 partitions
4. **Idempotent Data Generation** - Skips regeneration if data already exists
5. **Lazy Loading** - 60M rows loaded only when `.compute()` is called
6. **Distributed Aggregations** - GroupBy operations computed in parallel across workers
7. **Out-of-Core Processing** - Memory stays bounded processing 30 GB with ~2-8 GB cluster
8. **HoloViews Visualizations** - Interactive plots render correctly in JupyterLab
9. **Hierarchical Drill-Down** - Zoom-triggered aggregation (service group → service → host)
10. **Datashader Integration** - Millions of points rendered efficiently with zoom/pan

### Loading rules (air-gap)

- **Always** use `dd.read_parquet` / `load_active_spans_ddf` for analytics.
- **Never** `pyarrow.dataset.to_table()` on multi‑GB span sets (kernel OOM).
- PyArrow is fine for **chunked writes** and **row-group filters inside Dask workers**.

### PyArrow Dataset Capabilities

| Feature | Description | Benefit |
|---------|-------------|---------|
| **Schema Inspection** | Read schema without loading data | Fast exploration of unknown datasets |
| **Predicate Pushdown** | Filters applied at Parquet row-group level | Only matching data read from S3 |
| **Column Projection** | Only requested columns deserialized | Reduced I/O and memory |
| **Scanner API** | Batch-by-batch iteration | Constant memory for any dataset size |
| **Dask Integration** | Filters passed to Arrow engine | Distributed + pushdown combined |

### Data Loading Approaches Compared

| Approach | Use Case | Memory Model |
|----------|----------|--------------|
| `dd.read_parquet()` | Distributed aggregations | Partitioned across workers |
| `ds.dataset().to_table()` | **Avoid on large data** | Pulls result into kernel → OOM |
| `ds.dataset().scanner()` | Streaming processing | Constant memory (batch-at-a-time) |
| `dd.read_parquet(filters=)` | Distributed + filtered | Best of both worlds |

### Drill-Down Visualization Pattern

The notebook demonstrates **zoom-triggered hierarchy** for exploring large OTel datasets:

| Zoom Level | Aggregation | Time Bucket | Use Case |
|------------|-------------|-------------|----------|
| Zoomed out (>1h) | Service groups | 10 min | Cluster-wide overview |
| Medium (5m-1h) | Individual services | 1 min | Service comparison |
| Zoomed in (<5m) | Hosts/pods | 10 sec | Incident investigation |

This pattern enables exploring from region/AZ-level down to individual spans without
loading the entire dataset into memory.

### Key Insight

**No bottlenecks** in the data path:

1. **S3 → Arrow**: Predicate pushdown means only relevant row groups are read
2. **Arrow → Dask**: Zero-copy handoff, data stays in Arrow format
3. **Dask Workers**: Stream partitions, compute partial aggregations, release memory
4. **Final Result**: Only aggregated summaries returned to client

This architecture processes datasets **orders of magnitude larger than cluster memory**.
